# PostgreSQL Image Read Benchmark

Equivalent of `read_tesT_2.py` for PostgreSQL.
Reads 100 k randomly sampled images from `bench_images` using
multi-threaded `psycopg2` batch queries.

Reads connection details from environment variables `DB_HOST`, `DB_NAME`,
`DB_USER`, `DB_PASSWORD`; falls back to the docker-compose defaults.

In [13]:
import os
import random
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count

import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TOTAL_IMAGES      = 1_000_000
PATCHES_PER_WORKER = 1_000
N_WORKERS         = cpu_count()
BATCH_SIZE        = 1024
TABLE_NAME        = 'bench_images'

print(f'Workers: {N_WORKERS}, Patches/worker: {PATCHES_PER_WORKER}, Total reads: {N_WORKERS * PATCHES_PER_WORKER:,}')

Workers: 32, Patches/worker: 1000, Total reads: 32,000


## Sample Random IDs

## Read Benchmark

In [14]:
def read_worker(args):
    dsn, total_images, n_patches, batch_size, seed, table_name = args
    rng = random.Random(seed)
    ids = rng.sample(range(total_images), n_patches)

    conn = psycopg2.connect(dsn)
    cur  = conn.cursor()
    misses = 0
    bytes_fetched = 0

    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        cur.execute(
            f'SELECT id, image_data FROM {table_name} WHERE id = ANY(%s)',
            (batch,)
        )
        rows = cur.fetchall()
        fetched_ids = set()
        for img_id, image_data in rows:
            fetched_ids.add(img_id)
            bytes_fetched += len(bytes(image_data))
        for img_id in batch:
            if img_id not in fetched_ids:
                misses += 1

    cur.close()
    conn.close()
    return misses, bytes_fetched


worker_args = [
    (DSN, TOTAL_IMAGES, PATCHES_PER_WORKER, BATCH_SIZE, i, TABLE_NAME)
    for i in range(N_WORKERS)
 ]

start_time = time.perf_counter()
total_misses = 0
total_bytes  = 0
total_reads  = N_WORKERS * PATCHES_PER_WORKER

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = [executor.submit(read_worker, args) for args in worker_args]
    for f in as_completed(futures):
        misses, b = f.result()
        total_misses += misses
        total_bytes  += b

elapsed = time.perf_counter() - start_time

print(f'Read {total_reads:,} images in {elapsed:.2f} seconds')
print(f'Throughput: {total_reads / elapsed:,.0f} reads/sec')
print(f'Data transferred: {total_bytes / 1024**2:.1f} MB  ({total_bytes / elapsed / 1024**2:.1f} MB/s)')
if total_misses:
    print(f'Misses: {total_misses}')


Read 32,000 images in 0.28 seconds
Throughput: 113,065 reads/sec
Data transferred: 113.6 MB  (401.4 MB/s)
